# Resolve one product

Mandatory: `main_text`, `country_code`  
Optional: `ean`, `retailer_name`

This notebook directly uses SerpAPI, Crawl4AI, and optional PCA LLM reasoning. No UI, API server, Docker, CLI, background job, polling, or event-loop patching.

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
os.chdir(ROOT)
load_dotenv(ROOT / ".env")
sys.path.insert(0, str(ROOT / "src"))

from product_url_finder import ProductInput, Settings, resolve_product

settings = Settings.from_env()
settings.validate()
print("Ready")
print(f"SERP budget: {settings.serp_calls}")
print(f"Crawl budget: {settings.crawl_candidates}")
print(f"PCA LLM enabled: {settings.llm_enabled}")


## Input

In [ ]:
product = ProductInput(
    main_text="PKM ME04 WACHSENDES CHAOS BOOSTER",
    country_code="CH",
    ean="196214141070",       # optional: use None when unavailable
    retailer_name=None,       # optional: use None when unavailable
    row_id="DEMO-001",
)


## Run

In [ ]:
result = await resolve_product(product, settings)
result

## Submission row

In [ ]:
import pandas as pd

submission = pd.DataFrame([{
    "MAIN_TEXT": result["main_text"],
    "COUNTRY": result["country_code"],
    "RETAILER": result["retailer_name"],
    "EAN": result["ean"],
    "CANDIDATE_URLS": " | ".join(result["candidate_urls"]),
    "PRODUCT_URL": result["product_url"],
    "CONFIDENCE": result["confidence"],
    "VALIDATION_STATUS": result["status"],
    "IDENTITY_STATUS": result["identity_status"],
    "RETAILER_CHECK": result["retailer_check"],
    "JUSTIFICATION": result["justification"],
    "ARTIFACT_DIR": result["artifact_dir"],
}])
submission
